In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor

def create_model(trial):
    # Hyperparameter optimization for hidden layer neurons
    n_neurons = trial.suggest_int('n_neurons', 1, 20)
    
    model = Sequential([
        Dense(n_neurons, input_dim=X.shape[1], activation='tanh'),
        Dense(1, activation='linear')
    ])
    
    # Levenberg-Marquardt equivalent in Keras
    optimizer = Adam(learning_rate=0.01)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    
    return model

def objective(trial):
    # Create and compile model
    model = create_model(trial)
    
    # Wrap Keras model for scikit-learn compatibility
    keras_regressor = KerasRegressor(build_fn=lambda: model, epochs=100, batch_size=32, verbose=0)
    
    # Perform cross-validation
    scores = cross_val_score(keras_regressor, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
    return np.mean(-scores)

# Preprocessing
y = df['Price']
X = df.drop('Price', axis=1)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Optimize hidden layer neurons
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Best number of neurons
best_neurons = study.best_params['n_neurons']
print(f"Best number of neurons: {best_neurons}")

# Final model with best neurons
final_model = Sequential([
    Dense(best_neurons, input_dim=X.shape[1], activation='tanh'),
    Dense(1, activation='linear')
])

optimizer = Adam(learning_rate=0.01)
final_model.compile(optimizer=optimizer, loss='mean_squared_error')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train final model
history = final_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate model
mse = final_model.evaluate(X_test, y_test, verbose=0)
print(f"Mean Squared Error: {mse}")